# Piplining Demonstration

In this demo, we return to the polynomial example in the [command fifo demo](https://sdrangan.github.io/hwdesign/docs/demos/fifoif).  As described there,
the IP will compute a polynomial function:

```python
y[k] = a[0] + a[1]*x[k] + ... + a[d-1]*x[k]**d
```



## Protocol and Message Definitions

We use an indentical message passing protocol as the the [polynomial example](../stream/poly_demo.ipynb) in the command response FIFO:


- The PS sends a `PolyCmdHdr` with the coefficients, `coeffs`, number of samples, `nsamp`, and a transaction ID, `tx_id`.
- The IP repsonds with a `PolyRespHdr` message with the transcation ID, `tx_id`.  This ensures that the IP is in sync and has received the data.
- The PS then sends a vector of samples `x`.
- The IP computes the output data vector `y`
- When complete the IP sends a `PolyRespFtr` message with an indication of the number of samples and any error code.

We will build the messages using the [Pysilicon data schemas](https://sdrangan.github.io/pysilicon/docs/guide/schema).

In [23]:
import sys
from enum import IntEnum
from pathlib import Path

import numpy as np

from pysilicon.build.build import CodeGenConfig
from pysilicon.build.streamutils import copy_streamutils
from pysilicon.hw.arrayutils import gen_array_utils, read_array, get_nwords
from pysilicon.hw.dataschema import DataArray, DataList, EnumField, FloatField, IntField
from pysilicon.toolchain import toolchain, run_vitis_hls_result
from pysilicon.utils.vcd import VcdParser
from pysilicon.utils.timing import TimingDiagram, SigTimingInfo

In [2]:

# Parameters
include_dir = "include"  # Directory for generated header files
word_bw_supported = [32, 64]  # Supported word bitwidths for the generated code
max_nsamp = 128  # Maximum number of samples for the input and output data arrays


# Element types
Float32 = FloatField.specialize(bitwidth=32, include_dir=include_dir)
TransId = IntField.specialize(bitwidth=16, signed=False, include_dir=include_dir)
Nsamp = IntField.specialize(bitwidth=16, signed=False, include_dir=include_dir)

# Error codes for polynomial evaluation results
class PolyError(IntEnum):
    NO_ERROR = 0
    TLAST_EARLY_CMD_HDR = 1  # TLAST was asserted before the full command header was received
    NO_TLAST_CMD_HDR = 2  # The full command header was received but TLAST was never asserted
    TLAST_EARLY_SAMP_IN = 3  # TLAST was asserted before all input samples were received
    NO_TLAST_SAMP_IN = 4  # All input samples were received but TLAST was never asserted
    WRONG_NSAMP = 5  # The number of samples received does not match the expected number
PolyErrorField = EnumField.specialize(enum_type=PolyError, include_dir=include_dir)


# Coefficient array for polynomial evaluation (4 coefficients for a cubic polynomial)
class CoeffArray(DataArray):
    ncoeffs = 4
    element_type = Float32
    static = True
    max_shape = (ncoeffs,)
    include_dir = include_dir


# Data structures for the command and response headers and footers
class PolyCmdHdr(DataList):
    elements = {
        "tx_id": {
            "schema": TransId,
            "description": "Transaction ID",
        },
        "coeffs": {
            "schema": CoeffArray,
            "description": "Polynomial coefficients",
        },
        "nsamp": {
            "schema": Nsamp,
            "description": "Number of samples",
        },
    }
    include_dir = include_dir


class PolyRespHdr(DataList):
    elements = {
        "tx_id": {
            "schema": TransId,
            "description": "Echo of the transaction ID sent in the command",
        },
    }
    include_dir = include_dir


class PolyRespFtr(DataList):
    elements = {
        "nsamp_read": {
            "schema": Nsamp,
            "description": "Number of samples returned in the response",
        },
        "error": {
            "schema": PolyErrorField,
            "description": "Error code indicating success or type of failure",
        },
    }
    include_dir = include_dir


# List of all schema classes to be included in the generated code
schema_classes = [
    PolyErrorField,
    CoeffArray,
    PolyCmdHdr,
    PolyRespHdr,
    PolyRespFtr,
]

## Generating the Include Files

As before, once the Data schemas are defined, we can auto-generate the C++ header files.  Header files are generated for each data schema.  The header files provide:

- A C++ data structure for the data.
- Serialization and deserialization methods to store and transfer that data over various interfaces including AXI4-Stream

Auto-generating the include files avoids having to manually align the Python and C++ versions of data as well as manually packing and unpacking data for specific bitwidths.  All of this is done automatically.  Details are given in [PySilicon auto-generation documentation](https://sdrangan.github.io/pysilicon/docs/guide/schema/codegen.html).  Inlcude files can be generated from the  `gen_include` method.  For example, running `PolyCmdHdr.gen_include(....)` will generate two header files:

- `poly_cmd_hdr.h`:  The main header for use in Vitis synthesizable code. This file defines the C++ data structure as well as templated serialization and deserialization routines.
- `poly_cmd_hdr_tb.h`:  A companion header for non-synthesizable Vitis code such as testbenches. This file adds routines for reading/writing files and JSON dumps.

The code below loops through the data schemas and generates all the relevant files.


In [24]:
# Make the include directory
include_dir = Path.cwd() / include_dir
include_dir.mkdir(exist_ok=True)

# Remove any existing header files in the include directory
for header_file in include_dir.glob("*.h"):
    header_file.unlink()

# Create a code generation configuration and generate the header files for all schema classes
cfg = CodeGenConfig(root_dir=Path.cwd(), util_dir=include_dir)

# Generate header files for all schema classes
for schema_class in schema_classes:
    out_path = schema_class.gen_include(cfg=cfg, word_bw_supported=word_bw_supported)
    print(f"generated {out_path}")


generated c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\include\poly_error.h
generated c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\include\coeff_array.h
generated c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\include\poly_cmd_hdr.h
generated c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\include\poly_resp_hdr.h
generated c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\include\poly_resp_ftr.h


We also have to generate various general streaming utilities as well as utilies for floating point values.  This generation is done with the following code

In [25]:
# Generate the float32 include file with array utilities for the supported word bitwidths
out_path = gen_array_utils(Float32, word_bw_supported=word_bw_supported, cfg=cfg)
print(f"generated {out_path}")

# Copy the streamutils header file to the root directory
copy_streamutils(cfg=cfg)

generated c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\include\float32_array_utils.h


('C:\\Users\\sdran\\Documents\\repos\\hwdesign\\demos\\pipeline\\include\\streamutils_hls.h',
 'C:\\Users\\sdran\\Documents\\repos\\hwdesign\\demos\\pipeline\\include\\streamutils_tb.h')

## Implementing the Vitis kernel

The Vitis kernel is mostly identical to the polynomial example in the command-response FIFO example, except here we store the data in arrays before performing the computation. This is not the most efficient solution, but will also us to examine the loop without having to consider the streaming interfaces in the compute loop.  The C code is as follows.  The `ARRAY_PARTITION` pragma divides the arrays in blocks that can be accessed in parallel.

```c
float x[max_nsamp];
float y[max_nsamp];
#pragma HLS ARRAY_PARTITION variable=x type=cyclic factor=UNROLL_FACTOR  dim=1
#pragma HLS ARRAY_PARTITION variable=y type=cyclic factor=UNROLL_FACTOR  dim=1

```

Once the data is in the local storage, we can run the compute loop:

```c
    compute_loop:  for (int i = 0; i < nsamp; ++i) {
#pragma HLS loop_tripcount min=1 max=max_nsamp
#if UNROLL_FACTOR > 1
#pragma HLS unroll factor=UNROLL_FACTOR        
#else
#pragma HLS pipeline II=1
#endif
        y[i] = eval_poly_horner(cmd_hdr.coeffs.data, x[i]);
    }
```

## TCL file

We synthesize designs by adjusting by adjusting the compile directive `UNROLL_FACTOR`.  This is done in the TCL file that loops over three values 

```tcl
set unroll_factors {1 2 4}
```


## Sweeping over the unroll factors

We can now run the C-synthesis on the different unroll factors.  In a command shell:

```bash
vitis-run --mode hls --tcl run_hls_poly.tcl
```

Alternatively, you can run the following python command, but it tends to be slow when run from a jupyter notebook.  Either way, running the command will produce three solutions in the directory, `poly_uf1`, `poly_uf2` and `poly_uf4` corresponding to the three unroll factors.

In [26]:
#run_vitis_hls_result("run_hls_poly.tcl", output_path="vitis_run.txt")

## Parsing the synthesis reports

Each solution produces a **synthesis report**, an XML file.  For example, for `UNROLL_FACTOR=1`, the synthesis report is available at:

```bash
poly_uf1/solution1/syn/report/csynth.xml
```

You can read this report to get an extensive information about the loop pipelining and resource allocation.  PySilicon provides a basic parser to extract the information.  Below is the information for `UNROLL_FACTOR=1`.  You can see that all loops (including the load, compute, and store) all achieve `II=1`.

In [27]:
from pysilicon.utils.csynthparse import CsynthParser
import pandas as pd
import os

# Parse the synthesis for UF=1 
sol_path = os.path.join(os.getcwd(), 'poly_uf1', 'solution1')
parser = CsynthParser(sol_path=sol_path)

# Get the latency and initiation interval
print('Latency and Initiation Interval:')
parser.get_loop_pipeline_info()
#print(parser.loop_df)
with pd.option_context("display.max_columns", None, "display.width", 300):
    print(parser.loop_df)


# Get the resources
print('\nResource Usage:')
parser.get_resources()
print(parser.res_df)

Latency and Initiation Interval:
                                                 TripCountMin  TripCountMax  LatencyMin  LatencyMax  PipelineII  PipelineDepth
poly_Pipeline_VITIS_LOOP_325_1:VITIS_LOOP_325_1           NaN           NaN         NaN         NaN           1              1
poly_Pipeline_VITIS_LOOP_317_1:VITIS_LOOP_317_1           NaN           NaN         NaN         NaN           1              2
poly_Pipeline_compute_loop:compute_loop                   1.0        1024.0        29.0      1052.0           1             30
poly_Pipeline_VITIS_LOOP_374_1:VITIS_LOOP_374_1           0.0        1024.0         0.0      1024.0           1              2
poly_Pipeline_VITIS_LOOP_77_1:VITIS_LOOP_77_1             NaN           NaN         NaN         NaN           1              1

Resource Usage:
                                BRAM_18K  DSP      FF    LUT  URAM
poly_Pipeline_VITIS_LOOP_325_1         0    0     135     73     0
poly_Pipeline_VITIS_LOOP_317_1         0    0     107 

We now look at the stats for the compute loop at the differente unroll factors.  We see that the latency decreases with increasing the unrolling.  But the resources proportionally increase.

In [33]:
poly_dirs = sorted(
    p for p in Path.cwd().iterdir()
    if p.is_dir() and p.name.startswith("poly_uf") and p.name[7:].isdigit()
)
 
print('Found the directories:')
for p in poly_dirs:
    print(p)
ufs = [int(p.name[7:]) for p in poly_dirs]

print(ufs)

Found the directories:
c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\poly_uf1
c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\poly_uf2
c:\Users\sdran\Documents\repos\hwdesign\demos\pipeline\poly_uf4
[1, 2, 4]


In [ ]:
loop_rows = {}
resource_rows = {}
available_resources = None

for uf in ufs:
    sol_path = os.path.join(os.getcwd(), f"poly_uf{uf}", "solution1")
    parser = CsynthParser(sol_path=sol_path)
    parser.get_loop_pipeline_info()
    parser.get_resources()
    compute_matches = parser.loop_df.loc[
        parser.loop_df.index.str.contains("compute_loop", na=False)
    ]

    # Check that there is exactly one matching row for the compute_loop
    if compute_matches.empty:
        raise KeyError(f"No compute_loop row found for UF={uf}")

    if "Total" not in parser.res_df.index:
        raise KeyError(f"No Total resource row found for UF={uf}")

    if "Available" not in parser.res_df.index:
        raise KeyError(f"No Available resource row found for UF={uf}")

    # Extract the compute row 
    loop_rows[uf] = compute_matches.iloc[0]
    resource_rows[uf] = parser.res_df.loc["Total"]
    available_row = parser.res_df.loc["Available"]

    if available_resources is None:
        available_resources = available_row
    elif not available_resources.equals(available_row):
        raise ValueError(f"Available resource row differs for UF={uf}")

# Create a combined DataFrame for the loop info and resource usage across all UFs
loop_uf_df = pd.DataFrame.from_dict(loop_rows, orient="index").sort_index()
loop_uf_df.index.name = "UF"
resource_uf_df = pd.DataFrame.from_dict(resource_rows, orient="index").sort_index()
resource_uf_df.index.name = "UF"
resource_uf_df.loc["Available"] = available_resources

with pd.option_context("display.max_columns", None, "display.width", 300):
    print("loop_uf_df:")
    print(loop_uf_df)
    print("\nresource_uf_df:")
    print(resource_uf_df)


loop_uf_df:
    TripCountMin  TripCountMax  LatencyMin  LatencyMax  PipelineII  PipelineDepth
UF                                                                               
1            1.0        1024.0        29.0      1052.0         1.0           30.0
2            1.0         512.0        30.0       541.0         1.0           30.0
4            1.0         256.0        30.0       285.0         1.0           30.0

resource_uf_df:
           BRAM_18K  DSP      FF    LUT  URAM
UF                                           
1                 4   15    1963   3206     0
2                 4   30    3425   5555     0
4                 8   60    6220  10107     0
Available       280  220  106400  53200     0
